# Fold-Based Split Pipeline – Train / Test / Holdout

This notebook produces three memory-efficient combined datasets by splitting the
auxiliary file on the `fold` column **before** joining with the master dataset.

| Split   | Folds    | Purpose                  |
|---------|----------|--------------------------|
| train   | 1–5      | XGBoost model training   |
| test    | 6–7      | In-sample evaluation     |
| holdout | 8–9–10   | Out-of-sample evaluation |

**Why this is better than loading everything at once:**
- Aux data is filtered *before* the join → only ~20–50 % of rows loaded per pass
- All numeric columns are downcast to `float32` / `int32` → 50 % smaller per column
- Combined DataFrames are released from memory after saving → `gc.collect()` between iterations

All business logic lives in `code/data_processing.py`.

---
## 1 · Environment setup

In [ ]:
import gc
import os
import sys
import pandas as pd

# Project root is the directory that contains this notebook
PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.json")

print(f"Project root : {PROJECT_ROOT}")
print(f"Config path  : {CONFIG_PATH}")

---
## 2 · Load configuration

In [ ]:
from data_processing import load_config

config = load_config(CONFIG_PATH)

# ── Global settings ───────────────────────────────────────────────────────────
BASE_DIR    = config["global"]["base_data_dir"]
MASTER_FILE = config["global"]["master_data"]
AUX_FILE    = config["global"]["aux_data_car_dep_fold"]
MODEL_YEAR  = config["global"]["model_year"]

# ── Preprocess settings ───────────────────────────────────────────────────────
JOIN_KEY     = config["preprocess"]["joinkey"]
QUALITY_DIR  = config["preprocess"]["quality_report_dir"]

# ── Split definitions ─────────────────────────────────────────────────────────
SPLITS = config["splits"]   # e.g. {"train": [1,2,3,4,5], "test": [6,7], "holdout": [8,9,10]}

print(f"Model year : {MODEL_YEAR}")
print(f"Base dir   : {BASE_DIR}")
print(f"Master     : {MASTER_FILE}")
print(f"Aux        : {AUX_FILE}")
print(f"Join key   : {JOIN_KEY}")
print()
print("Splits:")
for name, folds in SPLITS.items():
    print(f"  {name:<10} → folds {folds}")

---
## 3 · Load master dataset once (float32 optimized)

In [ ]:
from data_processing import load_master_data_optimized, get_memory_usage

print("Loading master dataset...")
master_df = load_master_data_optimized(BASE_DIR, MASTER_FILE)

print(f"✅ master loaded  →  {master_df.shape[0]:,} rows × {master_df.shape[1]} columns")
print(f"   Memory usage   :  {get_memory_usage(master_df)}")
print()
print("master dtypes (first 10 cols):")
print(master_df.dtypes.head(10).to_string())
master_df.head(3)

---
## 4 · Process each split – filter aux → join → save

Each iteration:
1. Filters the aux file to the split's folds (only those rows enter RAM)
2. Downcasts aux to `float32` / `int32`
3. Inner-joins with master
4. Saves `<split>_combined.parquet` and `quality_report_<split>.csv`
5. Frees memory with `gc.collect()` before the next split

In [ ]:
from data_processing import (
    load_aux_data_by_folds,
    join_data,
    save_split_data,
)

# Accumulate summary info across all splits for the final report
summary_records = []

for split_name, fold_list in SPLITS.items():
    print("=" * 65)
    print(f"  SPLIT: {split_name.upper()}   (folds {fold_list})")
    print("=" * 65)

    # ── Step 1: load + filter aux ─────────────────────────────────────────
    print(f"[1/4] Loading aux filtered to folds {fold_list} ...")
    aux_df = load_aux_data_by_folds(BASE_DIR, AUX_FILE, fold_list)
    print(f"      aux shape   : {aux_df.shape[0]:,} rows × {aux_df.shape[1]} columns")
    print(f"      aux memory  : {get_memory_usage(aux_df)}")

    # ── Step 2: inner join ────────────────────────────────────────────────
    print(f"[2/4] Joining with master on '{JOIN_KEY}' ...")
    combined_df, drop_report = join_data(master_df, aux_df, JOIN_KEY, join_type="inner")
    print(f"      combined    : {combined_df.shape[0]:,} rows × {combined_df.shape[1]} columns")
    print(f"      memory      : {get_memory_usage(combined_df)}")

    if drop_report["dropped_from_master"] == 0 and drop_report["dropped_from_aux"] == 0:
        print("      ✅ PERFECT JOIN – no rows dropped")
    else:
        print(f"      ⚠️  Dropped from master : {drop_report['dropped_from_master']:,}  "
              f"({drop_report['pct_dropped_master']:.4f}%)")
        print(f"      ⚠️  Dropped from aux    : {drop_report['dropped_from_aux']:,}  "
              f"({drop_report['pct_dropped_aux']:.4f}%)")

    # ── Step 3: free aux immediately ─────────────────────────────────────
    del aux_df
    gc.collect()

    # ── Step 4: save parquet + quality report ─────────────────────────────
    print(f"[3/4] Saving '{split_name}_combined.parquet' and quality report ...")
    parquet_path, report_path = save_split_data(
        combined_df, BASE_DIR, split_name, QUALITY_DIR
    )
    print(f"      ✅ Parquet : {parquet_path}")
    print(f"      ✅ Report  : {report_path}")

    # ── Accumulate summary ────────────────────────────────────────────────
    summary_records.append(
        {
            "split":            split_name,
            "folds":            str(fold_list),
            "master_rows":      drop_report["master_rows"],
            "aux_rows":         drop_report["aux_rows"],
            "combined_rows":    drop_report["combined_rows"],
            "dropped_master":   drop_report["dropped_from_master"],
            "pct_drop_master":  drop_report["pct_dropped_master"],
            "dropped_aux":      drop_report["dropped_from_aux"],
            "pct_drop_aux":     drop_report["pct_dropped_aux"],
            "parquet_path":     parquet_path,
            "report_path":      report_path,
        }
    )

    # ── Step 4: free combined before next iteration ───────────────────────
    print(f"[4/4] Releasing memory ...")
    del combined_df
    gc.collect()
    print()

print("All splits processed! ✅")

---
## 5 · Pipeline summary

In [ ]:
summary_df = pd.DataFrame(summary_records)

print("╔══════════════════════════════════════════════════════════════╗")
print("║         FOLD-BASED SPLIT PIPELINE COMPLETE                   ║")
print(f"║  Model year : {MODEL_YEAR}                                         ║")
print("╠══════════════════════════════════════════════════════════════╣")
for rec in summary_records:
    print(f"║  {rec['split'].upper():<10}  folds {rec['folds']:<18} "
          f"→ {rec['combined_rows']:>10,} rows  ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Quality reports → {BASE_DIR}/{QUALITY_DIR}/")
print("╚══════════════════════════════════════════════════════════════╝")
print()

display(
    summary_df[
        ["split", "folds", "master_rows", "aux_rows", "combined_rows",
         "dropped_master", "pct_drop_master", "dropped_aux", "pct_drop_aux"]
    ].set_index("split")
)

---
## 6 · Verify output files exist on disk

In [ ]:
print("Output file verification:")
print()
all_ok = True
for rec in summary_records:
    for label, path in [("Parquet", rec["parquet_path"]), ("Report", rec["report_path"])]:
        exists = os.path.isfile(path)
        size   = f"{os.path.getsize(path) / 1024**2:.1f} MB" if exists else "–"
        status = "✅" if exists else "❌"
        print(f"  {status}  [{rec['split']:<8}] {label:<8} {size:<10}  {path}")
        if not exists:
            all_ok = False

print()
if all_ok:
    print("All output files confirmed on disk. ✅")
else:
    print("⚠️  One or more output files are missing – review errors above.")